# Demo 7 — Skeleton-of-Thought: The Same Tokens, 4x Faster
## Session 6: Advanced Prompt Engineering (Optional / Extra)

> **MLflow as the tape recorder.** Every prompt sent, every response returned, every parallel section's I/O, every wall-clock measurement — captured automatically by `mlflow.openai.autolog()` and the `@mlflow.trace` decorators below. Run the cells, then open the MLflow UI to replay every input and output the model saw, and watch the parallel waterfall.

**Watch this:** the same architecture review takes **~15 seconds sequentially**, then **~4 seconds when we fan out the sections in parallel.** Same tokens. Same output. **~4x faster wall-clock.**

The trick: ask for a **skeleton** first (one cheap call), then fire the section expansions **in parallel** with `asyncio.gather`. You replace a *sum* of section latencies with a *max*. That's the entire idea.

**Citation:** Ning et al. 2023, *Skeleton-of-Thought: Prompting LLMs for Efficient Parallel Generation* — [arXiv:2307.15337](https://arxiv.org/abs/2307.15337) (ICLR 2024). Headline: **2.39x average end-to-end wall-clock speedup across 12 LLMs**, with quality equal or better in 60% of cases. The SoT-R router variant reaches 2.01x with zero quality loss.

**The pitch in one line:** SoT is one of the few "free wins" in prompt engineering — you don't pay more tokens, you just orchestrate them better.

**Dependencies:** `openai`, `mlflow >= 3.10`, `matplotlib`. Local MLflow server on `http://127.0.0.1:5000`.

**Cost when real:** ~$0.02 per full run (`gpt-4o-mini`).

## Setup — env-var driven config

Everything below is configured from environment variables. Set `OPENAI_API_KEY` (or `OPENAI_API_KEY`) for live calls; leave it unset for mock mode. Override `MLFLOW_TRACKING_URI`, `MLFLOW_EXPERIMENT`, `OPENAI_BASE_URL`, `MODEL` to point at any OpenAI-compatible endpoint (OpenAI, Azure, vLLM, Ollama, Together, etc.) and any MLflow server.

```bash
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000
export OPENAI_API_KEY=sk-...
export MODEL=gpt-4o-mini
```

**Local-by-default.** Notebook ships with `OPENAI_BASE_URL=http://localhost:11434/v1` and `MODEL=qwen2.5-coder:7b` (Ollama). Set the env vars to point at OpenAI, Anthropic, vLLM, Together, etc. — any OpenAI-compatible endpoint.

**Cost tracking uses hypothetical local-LLM rates** (~$0.00005/1K tokens) so the cost-aware traces and scorers still produce meaningful numbers in the demos. Override the `PRICING` dict with your real per-1K rates for production.

In [ ]:
import os
import re
import json
import time
import asyncio
import random
import statistics
from typing import Optional

import mlflow
from mlflow.entities import SpanType
from openai import OpenAI, AsyncOpenAI

# --- MLflow tracking ---------------------------------------------------------
MLFLOW_TRACKING_URI      = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")  # optional (Databricks/remote)
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")  # optional
MLFLOW_REGISTRY_URI      = os.getenv("MLFLOW_REGISTRY_URI", MLFLOW_TRACKING_URI)
MLFLOW_EXPERIMENT        = os.getenv("MLFLOW_EXPERIMENT", "session6/demo_07_sot_parallel")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_registry_uri(MLFLOW_REGISTRY_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# --- LLM (OpenAI-compatible: OpenAI, Azure, vLLM, Ollama, Together, etc.) ---
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")  # Ollama default
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY", "ollama")  # Ollama ignores; any non-empty string works
MODEL    = os.getenv("MODEL", "qwen2.5-coder:7b")  # local-default for SoT parallel fan-out

# Reproducibility default. Both Sequential and SoT paths use this — that way the
# comparison is FAIR: same temperature, same prompt body, only the orchestration differs.
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))

# Per-1K-token pricing (USD). Update at each model release.
PRICING = {
    "gpt-4o":             {"in": 0.0025,   "out": 0.010},
    "gpt-4o-mini":        {"in": 0.00015,  "out": 0.0006},
    "o1-mini":            {"in": 0.003,    "out": 0.012},
    "claude-sonnet-4-6":  {"in": 0.003,    "out": 0.015},
    # Hypothetical local-LLM pricing (electricity + amortised GPU per 1K tokens — adjust to your infra)
    "qwen2.5-coder:14b":  {"in": 0.00005,  "out": 0.00005},
    "qwen2.5-coder:7b":   {"in": 0.00002,  "out": 0.00002},
    "llama3.1:8b":        {"in": 0.00002,  "out": 0.00002},
}

def price_of(model: str, usage) -> float:
    """USD cost for a single chat call. 0.0 in mock mode (no usage).
    Unknown models fall back to hypothetical local-LLM rates (~$0.00005/1K)."""
    if usage is None:
        return 0.0
    p = PRICING.get(model, {"in": 0.00005, "out": 0.00005})  # hypothetical fallback
    return usage.prompt_tokens / 1000 * p["in"] + usage.completion_tokens / 1000 * p["out"]


def tag_cost_latency(latency_ms: float, cost_usd: float) -> None:
    """Attach cost + latency to active span and root trace."""
    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute("latency_ms", latency_ms)
        span.set_attribute("cost_usd", cost_usd)
    try:
        mlflow.update_current_trace(tags={
            "cost_usd":   f"{cost_usd:.6f}",
            "latency_ms": f"{latency_ms:.0f}",
        })
    except Exception:
        pass


# Both sync and async clients — SoT needs AsyncOpenAI for asyncio.gather() fan-out.
client       = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY or "no-key")
async_client = AsyncOpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY or "no-key")

# --- Turn MLflow into the tape recorder -------------------------------------
# Captures every chat.completions.create call (sync + async) — request, response, tokens, latency.
mlflow.openai.autolog()

USE_MOCK = not OPENAI_API_KEY
print(f"MLflow:  {MLFLOW_TRACKING_URI}  (exp: {MLFLOW_EXPERIMENT})")
print(f"LLM:     {OPENAI_BASE_URL}  model={MODEL}  mock={USE_MOCK}  temperature={TEMPERATURE}")
print(f"UI:      open {MLFLOW_TRACKING_URI}  -> Experiments -> {MLFLOW_EXPERIMENT}")
print(f"NOTE: defaults to local Ollama (http://localhost:11434/v1). Set OPENAI_BASE_URL/OPENAI_API_KEY/MODEL to point at OpenAI/Anthropic/vLLM.")

if USE_MOCK:
    print("\nWARNING: no OPENAI_API_KEY / OPENAI_API_KEY set - running in MOCK mode.")
    print("         Section expansions are stubbed with a 3s deterministic sleep each.")
    print("         Sequential ~= 15s, Parallel ~= 3s. The speedup is the show.")

## The task — review a real architecture

We're reviewing the design of a new payments microservice. The review must cover **5 sections**: scalability, security, observability, cost, failure modes.

These sections are **independent** — security doesn't depend on cost, observability doesn't depend on scalability. That independence is exactly the precondition for SoT. (When sections cross-reference each other, SoT breaks — we'll discuss that at the end.)

Below is the brief the model will review. It's deliberately concrete — gRPC, Postgres, Kafka, K8s on EKS, with growth targets. The kind of one-pager you'd actually get on a design-review Slack thread.

In [ ]:
# -- The architecture under review -----------------------------------------

ARCHITECTURE_BRIEF = """
Service: `payments-core`. New microservice handling card authorization, capture, and refund.
API: gRPC (proto3) behind an Envoy edge proxy. Streaming endpoints for webhook fan-out.
Storage: Postgres 15 (multi-AZ, primary + 1 read replica). Ledger table is append-only.
Eventing: Kafka (3-broker MSK cluster) for `payment.events.*` topics. At-least-once delivery.
Compute: K8s on EKS, 6 nodes (m6i.2xlarge), HPA targeting 60% CPU.
Traffic: 5,000 RPS at launch, projected 50,000 RPS within 18 months (10x growth).
Auth: mTLS service-to-service; OAuth2 client-credentials for partner ingress.
Secrets: AWS Secrets Manager, rotated via Lambda every 30 days.
Deploys: ArgoCD GitOps, canary 5% -> 25% -> 100% with auto-rollback on SLO breach.
Region: single region (us-east-1) for v1; multi-region planned for v2.
""".strip()

SECTIONS = ["scalability", "security", "observability", "cost", "failure modes"]

print(ARCHITECTURE_BRIEF)
print()
print(f"Sections to cover: {SECTIONS}")

## Baseline: sequential — one big prompt, all 5 sections in one shot

The naive way. One call. The model writes the whole review top-to-bottom, autoregressively. Every section's tokens wait for the previous section's tokens to finish.

This is what most people do. It works. It's just **slow**.

In [ ]:
# -- Sequential baseline: one call, all sections in one autoregressive pass

SEQ_PROMPT = """You are reviewing the architecture below. Cover these 5 sections FULLY,
in order, with 3-5 sentences each: scalability, security, observability, cost, failure modes.

Architecture:
{brief}

Use `## <section>` headers. Be concrete - call out specific risks tied to the design."""


@mlflow.trace(
    name="sequential_review",
    span_type=SpanType.LLM,
    attributes={"technique": "sequential"},
)
async def sequential_review(brief: str) -> dict:
    span = mlflow.get_current_active_span()
    prompt = SEQ_PROMPT.format(brief=brief)
    if span:
        span.set_inputs({"prompt": prompt, "brief": brief,
                         "params": {"model": MODEL, "temperature": TEMPERATURE, "max_tokens": 1200}})

    t0 = time.time()
    if USE_MOCK:
        # Mock: simulate one big autoregressive call - 5 sections * ~3s each = 15s.
        await asyncio.sleep(15)
        text = "\n\n".join(
            f"## {s}\n[mock] Concrete risks for {s} given gRPC + Postgres + Kafka on EKS at 5k -> 50k RPS."
            for s in SECTIONS
        )
        tokens = 850
        latency_ms = (time.time() - t0) * 1000
        tag_cost_latency(latency_ms, 0.0)
    else:
        resp = await async_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,   # same temp on both paths — fair comparison
            max_tokens=1200,
        )
        latency_ms = (time.time() - t0) * 1000
        cost = price_of(MODEL, resp.usage)
        tag_cost_latency(latency_ms, cost)
        text = resp.choices[0].message.content
        tokens = resp.usage.completion_tokens if resp.usage else 0
    if span:
        span.set_outputs({"text": text, "tokens": tokens})
    return {"text": text, "tokens": tokens}


print("Running sequential review...")
t0 = time.time()
seq_result = await sequential_review(ARCHITECTURE_BRIEF)
t_seq = time.time() - t0

mlflow.update_current_trace(tags={
    "wall_clock_s": f"{t_seq:.2f}",
    "technique": "sequential",
    "tokens": str(seq_result["tokens"]),
})

print(f"\nSequential wall-clock: {t_seq:.2f}s")
print(f"Tokens (completion):   {seq_result['tokens']}")
print("\n--- Output (first 400 chars) ---")
print(seq_result["text"][:400] + "...")

## Skeleton-of-Thought — skeleton + parallel fan-out

**Two stages:**

1. **Skeleton call** — ask the model for a short numbered list of section headers. Cheap. Sequential.
2. **Parallel expansions** — fire one `async` call per section, **all at the same time**, with `asyncio.gather`. Concatenate.

The math:

```
t_sequential = sum(latency(section_i))                # = ~15s for us
t_sot        = latency(skeleton) + max(latency(section_i))   # = ~3s + small skeleton
```

**Sum becomes max.** That's the whole trick.

In [ ]:
# -- Stage 1: skeleton call ------------------------------------------------

SKELETON_PROMPT = """You will review the architecture below. First, output ONLY a numbered
skeleton of the section headers - one per line, 5 sections total: scalability, security,
observability, cost, failure modes.

Format EXACTLY:
1. <header>
2. <header>
...

No prose. No expansion. Headers only.

Architecture:
{brief}"""


@mlflow.trace(
    name="sot_skeleton",
    span_type=SpanType.LLM,
    attributes={"technique": "sot", "stage": "skeleton"},
)
async def get_skeleton(brief: str) -> list[str]:
    span = mlflow.get_current_active_span()
    prompt = SKELETON_PROMPT.format(brief=brief)
    if span:
        span.set_inputs({"prompt": prompt,
                         "params": {"model": MODEL, "temperature": TEMPERATURE, "max_tokens": 120}})

    t0 = time.time()
    if USE_MOCK:
        await asyncio.sleep(0.3)
        text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(SECTIONS))
        latency_ms = (time.time() - t0) * 1000
        tag_cost_latency(latency_ms, 0.0)
    else:
        resp = await async_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
            max_tokens=120,
        )
        latency_ms = (time.time() - t0) * 1000
        cost = price_of(MODEL, resp.usage)
        tag_cost_latency(latency_ms, cost)
        text = resp.choices[0].message.content

    # Parse "1. foo\n2. bar\n..."
    sections = []
    for line in text.splitlines():
        m = re.match(r"\s*\d+\.\s*(.+)", line)
        if m:
            sections.append(m.group(1).strip())
    if span:
        span.set_outputs({"text": text, "sections": sections})
    return sections


# Quick check
_test_skeleton = await get_skeleton(ARCHITECTURE_BRIEF)
print("Skeleton parsed:")
for i, s in enumerate(_test_skeleton, 1):
    print(f"  {i}. {s}")

In [ ]:
# -- Stage 2: per-section async expansion ----------------------------------

EXPAND_PROMPT = """You are reviewing the architecture below. Write the `## {section}` section
of the review ONLY. 3-5 sentences. Concrete risks tied to the design choices.
Do NOT reference or mention any other section.

Architecture:
{brief}"""


@mlflow.trace(
    name="sot_expand",
    span_type=SpanType.LLM,
    attributes={"technique": "sot", "stage": "expand"},
)
async def expand_section(brief: str, section: str) -> dict:
    span = mlflow.get_current_active_span()
    prompt = EXPAND_PROMPT.format(brief=brief, section=section)
    if span:
        span.set_inputs({"section": section, "prompt": prompt,
                         "params": {"model": MODEL, "temperature": TEMPERATURE, "max_tokens": 300}})

    t0 = time.time()
    if USE_MOCK:
        # Each expansion blocks for ~3s. In parallel, total wall-clock is ~3s, not 15s.
        await asyncio.sleep(3)
        text = (
            f"## {section}\n[mock] Concrete risks for {section} given gRPC + Postgres + Kafka "
            f"on EKS at 5k -> 50k RPS: specific failure modes, mitigations, and SLO impact."
        )
        tokens = 160
        latency_ms = (time.time() - t0) * 1000
        tag_cost_latency(latency_ms, 0.0)
    else:
        resp = await async_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,   # same temp on both paths — fair comparison
            max_tokens=300,
        )
        latency_ms = (time.time() - t0) * 1000
        cost = price_of(MODEL, resp.usage)
        tag_cost_latency(latency_ms, cost)
        text = resp.choices[0].message.content
        tokens = resp.usage.completion_tokens if resp.usage else 0
    if span:
        span.set_outputs({"section": section, "text": text, "tokens": tokens})
    return {"section": section, "text": text, "tokens": tokens}


# -- Stage 3: orchestrator -------------------------------------------------

@mlflow.trace(
    name="sot_review",
    span_type=SpanType.CHAIN,
    attributes={"technique": "sot"},
)
async def sot_review(brief: str) -> dict:
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"brief": brief, "model": MODEL, "temperature": TEMPERATURE})

    sections = await get_skeleton(brief)
    # The whole point: fire all expansions concurrently with asyncio.gather.
    expansions = await asyncio.gather(*[expand_section(brief, s) for s in sections])
    body = "\n\n".join(e["text"] for e in expansions)
    total_tokens = sum(e["tokens"] for e in expansions)
    result = {"text": body, "tokens": total_tokens, "n_sections": len(sections), "sections": sections}
    if span:
        span.set_outputs({"text": body, "tokens": total_tokens,
                          "n_sections": len(sections), "sections": sections})
    return result


print("Running SoT review (skeleton + parallel fan-out)...")
t0 = time.time()
sot_result = await sot_review(ARCHITECTURE_BRIEF)
t_sot = time.time() - t0   # total wall-clock for the SoT path

mlflow.update_current_trace(tags={
    "wall_clock_s": f"{t_sot:.2f}",
    "n_sections": str(sot_result["n_sections"]),
    "technique": "sot",
    "tokens": str(sot_result["tokens"]),
})

print(f"\nSoT wall-clock:       {t_sot:.2f}s   (total wall-clock measured end-to-end)")
print(f"Tokens (completion):  {sot_result['tokens']}")
print(f"Sections:             {sot_result['sections']}")
print("\n--- Output (first 400 chars) ---")
print(sot_result["text"][:400] + "...")

## The headline number

Side by side. The tokens should be roughly equal — that's the whole pitch. **Same work, just orchestrated better.**

In [ ]:
# -- The headline number ---------------------------------------------------

speedup = t_seq / t_sot

print("=" * 60)
print("  WALL-CLOCK SHOWDOWN")
print("=" * 60)
print(f"  Sequential : {t_seq:6.2f}s   ({seq_result['tokens']} completion tokens)")
print(f"  SoT        : {t_sot:6.2f}s   ({sot_result['tokens']} completion tokens)")
print(f"  Speedup    : {speedup:6.2f}x")
print("=" * 60)
print()
print(f"  Token ratio (SoT / Sequential): {sot_result['tokens'] / max(seq_result['tokens'], 1):.2f}")
print("  -> tokens roughly equal; the win is pure orchestration.")
print()
print(f"  Ning et al. 2024 reported a 2.39x average across 12 LLMs. Ours: {speedup:.2f}x.")

## Open the MLflow UI — the waterfall makes it visceral

1. Visit **http://127.0.0.1:5000** -> experiment **`session6/demo_07_sot_parallel`** -> Traces.
2. Open the latest **`sequential_review`** trace. You'll see **one big LLM span** that ran for ~15 seconds. One bar. Long.
3. Open the latest **`sot_review`** trace. You'll see:
   - `sot_review` (CHAIN) as the root
   - `sot_skeleton` (LLM) — tiny bar at the start, ~0.3s
   - **Five `sot_expand` (LLM) spans stacked in parallel** — same start time, all finishing in ~3s
4. The waterfall view shows the latency win as a **shape**: tall-and-narrow (SoT) vs short-and-wide (sequential). That visual is the whole reason this technique exists.

Filter past runs by technique: `tags.technique = 'sot'` in the trace search bar.

In [ ]:
# -- Multi-trial bake-off: 3 runs each, distribution + mean speedup --------

N_TRIALS = 3
seq_times: list[float] = []
sot_times: list[float] = []

print(f"Running {N_TRIALS} trials of each technique...\n")

for i in range(N_TRIALS):
    t0 = time.time()
    await sequential_review(ARCHITECTURE_BRIEF)
    seq_times.append(time.time() - t0)

    t0 = time.time()
    await sot_review(ARCHITECTURE_BRIEF)
    sot_times.append(time.time() - t0)

    print(f"  Trial {i+1}: seq={seq_times[-1]:5.2f}s  sot={sot_times[-1]:5.2f}s  speedup={seq_times[-1]/sot_times[-1]:.2f}x")

mean_seq = statistics.mean(seq_times)
mean_sot = statistics.mean(sot_times)
mean_speedup = mean_seq / mean_sot

print()
print("=" * 60)
print(f"  Mean sequential : {mean_seq:.2f}s   (stdev {statistics.pstdev(seq_times):.2f})")
print(f"  Mean SoT        : {mean_sot:.2f}s   (stdev {statistics.pstdev(sot_times):.2f})")
print(f"  Mean speedup    : {mean_speedup:.2f}x")
print("=" * 60)

# Quick text-art distribution since matplotlib may not always be available
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.boxplot([seq_times, sot_times], labels=["sequential", "SoT"], vert=False)
    ax.set_xlabel("wall-clock seconds")
    ax.set_title(f"Wall-clock distribution over {N_TRIALS} trials (mean speedup {mean_speedup:.2f}x)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("(matplotlib not installed - skipping plot)")

## When SoT does NOT help

SoT is not magic. Three regimes where it's the wrong call:

1. **Cross-referencing sections.** "As discussed above in point 2…" — parallel expansions can't satisfy back-references because each call only sees the skeleton, not the other expansions. Inter-section duplication and contradiction become the failure mode.
2. **Short outputs (<300 completion tokens).** The skeleton call's own latency dominates. You pay the overhead, you get no parallelism win.
3. **Single-section / atomic answers.** "What is the p99 latency?" doesn't decompose. Don't fan out one thing.

**Solution: SoT-R (router variant).** Ning et al. ship a classifier that decides per-question whether SoT applies. Sequential answers skip SoT entirely. Reaches **2.01x average speedup with zero quality loss** (vs 2.39x with occasional quality loss for the unconditional version). In production, this is the variant you want.

## The production angle — why this matters for chat UX

**User-perceived latency is the metric that matters**, not token count. A 4x reduction in wall-clock time is a 4x reduction in how long the user sits staring at a spinner. Every UX study on chat interfaces says the same thing: time-to-first-useful-content drives satisfaction.

SoT is one of the rare prompt-engineering wins that costs **nothing extra**:

- Tokens: unchanged (same model, same output volume)
- Quality: equal or better in 60% of cases per the paper
- Latency: divided by `~N_sections / 1`

Pair SoT with **streaming** on each parallel call and you get a second perceived-latency win: the user starts reading section 1 while sections 2-5 are still generating. The combination is what production assistants like Perplexity and Claude.ai's research mode actually ship under the hood.

## Takeaways

- **Independent sections = SoT candidate.** If the parts of your answer don't reference each other, you're leaving wall-clock latency on the table by generating them sequentially.
- **Tokens unchanged, latency divided.** SoT replaces a *sum* over section latencies with a *max*. Same cost, different orchestration.
- **Mind the inter-section references.** Cross-references break parallel expansion. Use SoT-R, or refactor your prompt to keep sections self-contained.
- **2.39x average per the paper (Ning et al. 2024).** Real measured speedup across 12 LLMs. Your mileage with `gpt-4o-mini` + 5 sections will land in the 3-5x range.

## ⚠️ Reasoning model caveat

This notebook assumes a **non-reasoning** base model. On `o1` / `o3` / Claude Extended Thinking / Gemini Thinking:
- Drop the CoT / ToT / Plan-and-Solve scaffolding — the model does it internally
- Use `developer` role instead of `system` on `o1-2024-12-17+`
- Never pass OpenAI `reasoning_effort` and Gemini `thinking_level` through the same OpenAI-compatible adapter
- See Block 5 of the slides for what's redundant vs what survives

SoT **still works on reasoning models** — the win is orchestration (sum → max wall-clock), not prompt scaffolding. The catch: reasoning models tend to take longer per call, so the skeleton-call overhead is proportionally larger. SoT remains a win whenever you have ≥ 3 truly independent sections.

## Replay every input + output in MLflow

Every prompt and every response from this demo is now in MLflow. Open the UI and click any trace to see the full I/O log.

In [ ]:
import IPython.display as D

ui = MLFLOW_TRACKING_URI.rstrip("/")
exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
exp_id = exp.experiment_id if exp else "0"
link = f"{ui}/#/experiments/{exp_id}?searchFilter=&orderByKey=attributes.start_time&orderByAsc=false"
D.display(D.Markdown(f"**Open the experiment in MLflow:** [{link}]({link})"))

recent = mlflow.search_traces(experiment_ids=[exp_id], max_results=5)
recent[["request_id", "timestamp_ms", "execution_time_ms", "tags"]] if not recent.empty else "No traces yet — run the cells above."